# Question 25 - 125. Valid Palindrome

A phrase is a **palindrome** if, after converting all uppercase letters into lowercase letters and removing all non-alphanumeric characters, it reads the same forward and backward. Alphanumeric characters include letters and numbers.

Given a string `s`, return `true` *if it is a **palindrome**, or* `false` *otherwise*.

**Example 1:**

    Input: s = "A man, a plan, a canal: Panama"
    Output: true
    Explanation: "amanaplanacanalpanama" is a palindrome.

**Example 2:**

    Input: s = "race a car"
    Output: false
    Explanation: "raceacar" is not a palindrome.

**Example 3:**

    Input: s = " "
    Output: true
    Explanation: s is an empty string "" after removing non-alphanumeric characters.
    Since an empty string reads the same forward and backward, it is a palindrome.

**Constraints:**

- `1 <= s.length <= 2 * 10^5`
- `s` consists only of printable ASCII characters.

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Two pointers converging from both ends (skip non-alphanumerics on the fly)

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (clean copy + reverse) | O(n) | O(n) |
| 2️⃣ Optimized (two pointers in place) | O(n) | O(1) |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding

🗣️ *"Two normalisation rules: ignore everything that isn't a letter or digit, and compare case-insensitively. Then check it reads the same both ways."*

- Digits count as alphanumeric? → **Yes** (`"0P"` → `false`, since `0 ≠ p`).
- After cleaning, an empty string is a palindrome → `true`.
- ASCII only, so `str.isalnum()` and `lower()` are safe.

### Step 1 · Brute force

**Idea:** Build the cleaned, lowercased string, then compare it with its reverse.

🗣️ *"The simplest correct approach filters to alphanumerics, lowercases, and compares with the reversed copy. It's O(n) time, but it allocates two extra strings of up to n characters. I can do the same comparison in place with two pointers."*

In [1]:
class SolutionBrute:
    def isPalindrome(self, s: str) -> bool:
        cleaned = "".join(ch.lower() for ch in s if ch.isalnum())
        return cleaned == cleaned[::-1]

### Step 2 · Optimize

**Bottleneck:** not time, but **space**. We materialise the cleaned string and its reverse just to compare characters pairwise.

**Key insight:** a palindrome check *is* a pairwise comparison of the i-th from the left with the i-th from the right. Put `left` at the start and `right` at the end. Skip over non-alphanumerics as you go, compare lowercased characters, and move both pointers inward. Stop when they cross.

🗣️ *"I'll use two pointers from both ends. Each pointer skips characters that aren't alphanumeric. When both are on valid characters, I compare them lowercased. A mismatch means false. Otherwise I move both inward. The pointers together cross the string once, so it's O(n) time with O(1) extra space."*

#### Choosing the data structure & algorithm: why this, why not that

| Option | Verdict | Why / why not |
|---|---|---|
| Clean + reverse + compare | ✅ (fine first answer) | O(n) time, very readable. O(n) extra memory. |
| Regex clean (`re.sub(r'[^a-z0-9]', '', s.lower())`) | ⚠️ | Same O(n) space, plus regex overhead. No advantage. |
| Stack: push the first half, pop to compare | ❌ | Still O(n) space, and you need the cleaned length first. |
| Recursion on `s[1:-1]` | ❌ | Slicing copies (O(n²) total) and risks recursion depth at 2·10⁵. |
| **Two pointers, skip in place** | ✅ (best) | O(n) time and O(1) space. No copies at all. |

In [2]:
class Solution:
    def isPalindrome(self, s: str) -> bool:
        left, right = 0, len(s) - 1
        while left < right:
            while left < right and not s[left].isalnum():
                left += 1
            while left < right and not s[right].isalnum():
                right -= 1
            if s[left].lower() != s[right].lower():
                return False
            left += 1
            right -= 1
        return True

### Step 3 · Toy example walkthrough

`s = "race a car"` (indices 0–9)

| left | s[left] | right | s[right] | action |
|---|---|---|---|---|
| 0 | r | 9 | r | equal → move in |
| 1 | a | 8 | a | equal → move in |
| 2 | c | 7 | c | equal → move in |
| 3 | e | 6 | ' ' | right skips the space → right = 5 ('a') |
| 3 | e | 5 | a | **e ≠ a** → `False` ✅ |

`s = "A man, a plan..."`: the pointers skip `,`, `:` and spaces, and compare `A`/`a` after lowercasing. Every pair matches until they cross → `True`.

**Why the `left < right` guard inside the skip loops?** Without it, a string like `".,"` could push a pointer past the other one or out of range.

In [3]:
cases = [
    ("A man, a plan, a canal: Panama", True),
    ("race a car", False),
    (" ", True),
    ("0P", False),
    ("a.", True),
    (".,", True),
    ("ab_a", True),
    ("Was it a car or a cat I saw?", True),
]
for s, expected in cases:
    assert SolutionBrute().isPalindrome(s) == expected
    assert Solution().isPalindrome(s) == expected
    print(repr(s), "->", expected)

import random
for _ in range(2000):
    s = "".join(random.choices("aAb1 ,.", k=random.randint(1, 8)))
    assert Solution().isPalindrome(s) == SolutionBrute().isPalindrome(s), s
print("All tests passed ✅")

'A man, a plan, a canal: Panama' -> True
'race a car' -> False
' ' -> True
'0P' -> False
'a.' -> True
'.,' -> True
'ab_a' -> True
'Was it a car or a cat I saw?' -> True
All tests passed ✅


### Step 4 · Wrap up

| | Time | Space | Why |
|---|---|---|---|
| Clean + reverse | O(n) | O(n) | Builds two new strings |
| **Two pointers** | **O(n)** | **O(1)** | Each index is visited by at most one pointer |

**Complexity, explained:** `left` only moves right and `right` only moves left, and they stop when they meet. So together they make at most n steps, even counting the inner skip loops. Each step is O(1) (`isalnum`, `lower`, compare). Only two integers are stored.

**Edge cases:** all punctuation (→ `True`), digits vs letters (`"0P"`), a single character, mixed case.

**Follow-up:** *Valid Palindrome II (LC 680): may delete at most one character?* → On the first mismatch, try skipping either `left` or `right` and check the remaining range with the same two-pointer loop. Still O(n).

🗣️ **30-second recap:** *"Two pointers from both ends. Each skips non-alphanumeric characters, then I compare the two characters lowercased. A mismatch returns false; otherwise both move inward until they meet. O(n) time and O(1) space, no cleaned copy needed."*